# Stage 3 – Spark ML: Trip-Level Fuel Consumption Forecasting

**Team:** team15  
**Target:** `fuel_l_per_100km` (L / 100 km)  
**Models:** Linear Regression · Gradient-Boosted Trees Regressor

## 0. Configuration

In [ ]:
import os

TEAM = "team15"
WAREHOUSE = "project/hive/warehouse"

TRIP_FEATURES_HDFS = "/user/team15/project/analytics/trip_features"

HDFS_DATA_TRAIN  = "project/data/train"
HDFS_DATA_TEST   = "project/data/test"
HDFS_MODEL1      = "project/models/model1"
HDFS_MODEL2      = "project/models/model2"
HDFS_PRED1       = "project/output/model1_predictions"
HDFS_PRED2       = "project/output/model2_predictions"
HDFS_EVAL        = "project/output/evaluation"

LOCAL_DATA_DIR   = "../data"
LOCAL_OUTPUT_DIR = "../output"
LOCAL_MODEL1_DIR = "../models/model1"
LOCAL_MODEL2_DIR = "../models/model2"

RANDOM_SEED    = 42
TRAIN_RATIO    = 0.7
TEST_RATIO     = 0.3
CV_FOLDS       = 3
CV_PARALLELISM = 4

TARGET = "fuel_l_per_100km"

NUMERIC_FEATURES = [
    "duration_min", "observed_seconds", "distance_km",
    "speed_mean", "speed_median", "speed_p95",
    "stop_go_ratio", "idle_time_min",
    "maf_mean", "maf_p95",
    "rpm_mean", "rpm_p95",
    "abs_load_mean", "oat_mean",
    "hv_current_mean", "hv_soc_mean", "hv_voltage_mean",
    "gen_weight", "eng_dis",
]

CATEGORICAL_FEATURES = [
    "vehtype", "vehclass", "transmission",
    "drive_wheels", "eng_type", "eng_conf",
]

def run(cmd):
    return os.popen(cmd).read()

print("Config OK")

## 1. SparkSession

In [ ]:
from pyspark.sql import SparkSession

ON_CLUSTER = bool(os.environ.get("HADOOP_CONF_DIR") or os.environ.get("YARN_CONF_DIR"))

if ON_CLUSTER:
    spark = (
        SparkSession.builder
        .appName("{} - spark ML".format(TEAM))
        .master("yarn")
        .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883")
        .config("spark.sql.warehouse.dir", WAREHOUSE)
        .config("spark.sql.avro.compression.codec", "snappy")
        .config("spark.sql.shuffle.partitions", "32")
        .enableHiveSupport()
        .getOrCreate()
    )
else:
    spark = (
        SparkSession.builder
        .appName("{} - spark ML [local]".format(TEAM))
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "|" , spark.sparkContext.master)

## 2. Read `trip_features`

In [ ]:
df = spark.read.parquet(TRIP_FEATURES_HDFS)
df.printSchema()
print("Total rows:", df.count())
df.show(5, truncate=False)

## 3. Select features & target

In [ ]:
from pyspark.sql import functions as F

existing_numeric     = [c for c in NUMERIC_FEATURES     if c in df.columns]
existing_categorical = [c for c in CATEGORICAL_FEATURES if c in df.columns]

print("Numeric    :", existing_numeric)
print("Categorical:", existing_categorical)

df_ml = (
    df.select(existing_numeric + existing_categorical + [TARGET])
    .where(F.col(TARGET).isNotNull())
    .where(F.col(TARGET) > 0)
    .withColumnRenamed(TARGET, "label")
)

print("Rows after filter:", df_ml.count())
df_ml.describe(["label"]).show()

## 4. Preprocessing pipeline

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler

imputer = Imputer(
    inputCols=existing_numeric,
    outputCols=["{}_imp".format(c) for c in existing_numeric],
    strategy="median",
)
imputed_numeric_cols = ["{}_imp".format(c) for c in existing_numeric]

indexers = [
    StringIndexer(inputCol=c, outputCol="{}_idx".format(c), handleInvalid="keep")
    for c in existing_categorical
]
encoders = [
    OneHotEncoder(inputCol="{}_idx".format(c), outputCol="{}_ohe".format(c), dropLast=True)
    for c in existing_categorical
]
ohe_cols = ["{}_ohe".format(c) for c in existing_categorical]

assembler = VectorAssembler(
    inputCols=imputed_numeric_cols + ohe_cols,
    outputCol="features_raw",
    handleInvalid="keep",
)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)

prep_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, scaler])
print("Pipeline stages:", len(prep_pipeline.getStages()))

## 5. Train / test split

In [ ]:
train_raw, test_raw = df_ml.randomSplit([TRAIN_RATIO, TEST_RATIO], seed=RANDOM_SEED)
print("Train:", train_raw.count(), "| Test:", test_raw.count())

prep_model = prep_pipeline.fit(train_raw)
train_data = prep_model.transform(train_raw).select("features", "label")
test_data  = prep_model.transform(test_raw).select("features", "label")

train_data.cache()
test_data.cache()

train_data.coalesce(1).write.mode("overwrite").format("json").save(HDFS_DATA_TRAIN)
test_data.coalesce(1).write.mode("overwrite").format("json").save(HDFS_DATA_TEST)

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
run("hdfs dfs -getmerge {src}/*.json {dst}/train.json".format(src=HDFS_DATA_TRAIN, dst=LOCAL_DATA_DIR))
run("hdfs dfs -getmerge {src}/*.json {dst}/test.json".format(src=HDFS_DATA_TEST,  dst=LOCAL_DATA_DIR))
print("Splits saved.")

## 6. Evaluators

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

## 7. Model 1 – Linear Regression

Grid: `elasticNetParam` ∈ {0.0, 0.5, 1.0} × `regParam` ∈ {0.01, 0.1, 1.0}

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=100)

param_grid_lr = (
    ParamGridBuilder()
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .addGrid(lr.regParam,        [0.01, 0.1, 1.0])
    .build()
)

cv_lr = CrossValidator(
    estimator=lr,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluator_rmse,
    numFolds=CV_FOLDS,
    parallelism=CV_PARALLELISM,
    seed=RANDOM_SEED,
)

cv_model_lr = cv_lr.fit(train_data)
model1 = cv_model_lr.bestModel

print("elasticNetParam:", model1.getElasticNetParam())
print("regParam       :", model1.getRegParam())

In [ ]:
predictions1 = model1.transform(test_data)

rmse1 = evaluator_rmse.evaluate(predictions1)
mae1  = evaluator_mae.evaluate(predictions1)
r2_1  = evaluator_r2.evaluate(predictions1)

print("RMSE={:.4f}  MAE={:.4f}  R2={:.4f}".format(rmse1, mae1, r2_1))
predictions1.select("label", "prediction").show(10)

In [ ]:
model1.write().overwrite().save(HDFS_MODEL1)
run("hdfs dfs -get {} {}".format(HDFS_MODEL1, LOCAL_MODEL1_DIR))

os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
(
    predictions1.select("label", "prediction")
    .coalesce(1).write.mode("overwrite")
    .format("csv").option("sep", ",").option("header", "true")
    .save(HDFS_PRED1)
)
run("hdfs dfs -getmerge {src}/*.csv {dst}/model1_predictions.csv".format(src=HDFS_PRED1, dst=LOCAL_OUTPUT_DIR))
print("model1 saved.")

## 8. Model 2 – Gradient-Boosted Trees

Grid: `maxDepth` ∈ {3, 5} × `stepSize` ∈ {0.05, 0.1}

In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(featuresCol="features", labelCol="label", maxIter=50, seed=RANDOM_SEED)

param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build()
)

cv_gbt = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_rmse,
    numFolds=CV_FOLDS,
    parallelism=CV_PARALLELISM,
    seed=RANDOM_SEED,
)

cv_model_gbt = cv_gbt.fit(train_data)
model2 = cv_model_gbt.bestModel

print("maxDepth:", model2.getMaxDepth())
print("stepSize:", model2.getStepSize())

In [ ]:
predictions2 = model2.transform(test_data)

rmse2 = evaluator_rmse.evaluate(predictions2)
mae2  = evaluator_mae.evaluate(predictions2)
r2_2  = evaluator_r2.evaluate(predictions2)

print("RMSE={:.4f}  MAE={:.4f}  R2={:.4f}".format(rmse2, mae2, r2_2))
predictions2.select("label", "prediction").show(10)

In [ ]:
model2.write().overwrite().save(HDFS_MODEL2)
run("hdfs dfs -get {} {}".format(HDFS_MODEL2, LOCAL_MODEL2_DIR))

(
    predictions2.select("label", "prediction")
    .coalesce(1).write.mode("overwrite")
    .format("csv").option("sep", ",").option("header", "true")
    .save(HDFS_PRED2)
)
run("hdfs dfs -getmerge {src}/*.csv {dst}/model2_predictions.csv".format(src=HDFS_PRED2, dst=LOCAL_OUTPUT_DIR))
print("model2 saved.")

## 9. Compare models

In [ ]:
eval_df = spark.createDataFrame(
    [
        (str(model1), float(rmse1), float(mae1), float(r2_1)),
        (str(model2), float(rmse2), float(mae2), float(r2_2)),
    ],
    ["model", "RMSE", "MAE", "R2"],
)
eval_df.show(truncate=False)

(
    eval_df.coalesce(1).write.mode("overwrite")
    .format("csv").option("sep", ",").option("header", "true")
    .save(HDFS_EVAL)
)
run("hdfs dfs -getmerge {src}/*.csv {dst}/evaluation.csv".format(src=HDFS_EVAL, dst=LOCAL_OUTPUT_DIR))

winner = "Model 1 (LR)" if rmse1 < rmse2 else "Model 2 (GBT)"
print("Best model by RMSE:", winner)

In [ ]:
spark.stop()